# 23.1 DINO 版本與官方預訓練特徵

[完整圖文教材](https://birdhackor.github.io/learn_to_yolo/lessons/23-dino-versions/)

先執行下一格的環境格，再閱讀、執行最後的完整實驗。各節互相獨立，不需要先跑其他節。

最後一格存著這段程式在 CPU 上實際執行的輸出；執行環境與其他實驗紀錄見網站的〈驗證範圍〉。範例不用預訓練權重，也不需要下載資料。

In [ ]:
# 全新 runtime 可由此格開始。只取得小型程式，不自動下載 LFS 資料。
import importlib.metadata
import os
from pathlib import Path
import subprocess
import sys

REF = 'lessons-v0.6.0'
base = Path("/content") if Path("/content").is_dir() else Path.cwd()
repository = base / ("learn_to_yolo_" + REF.replace(".", "_").replace("-", "_"))
if not repository.exists():
    environment = os.environ.copy()
    environment["GIT_LFS_SKIP_SMUDGE"] = "1"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", REF,
                    "https://github.com/birdhackor/learn_to_yolo.git", str(repository)],
                   env=environment, check=True)
actual_ref = subprocess.check_output(
    ["git", "-C", str(repository), "describe", "--tags", "--exact-match"], text=True
).strip()
if actual_ref != REF:
    raise RuntimeError("程式版本不同，請使用新的 runtime 或移除舊 clone 後重試。")

# 各節實驗都在 CPU 上執行，執行紀錄用的是 PyTorch 2.9.1。已經是 2.9.1 就沿用（CPU 或 CUDA 版都可以）；
# 其他版本會改裝成 2.9.1 的 CPU 版，之後這個工作階段就用不到 GPU，但各節實驗本來就不需要。
try:
    torch_version = importlib.metadata.version("torch").split("+")[0]
except importlib.metadata.PackageNotFoundError:
    torch_version = None
if torch_version != "2.9.1":
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.9.1",
                    "--index-url", "https://download.pytorch.org/whl/cpu"], check=True)
    if "torch" in sys.modules:
        raise RuntimeError("PyTorch 已更新：請從選單重新啟動工作階段，再由第一格執行。")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                *['numpy==2.3.5', 'pillow==12.0.0', 'matplotlib==3.10.7']], check=True)
os.chdir(repository)
sys.path.insert(0, str(repository))
print("固定教材版本：", REF)
print("目前目錄：", repository)


## 本節可修改的完整實驗

先預測結果，再執行；確認輸出後，試做網頁的自主練習。

### 可選：官方 DINOv2 凍結特徵

下面的三向量主例不下載權重。跑完後可另開一個 code cell：

```python
!python scripts/run_dino_pretrained.py --report artifacts/runs/dinov2-pretrained/report.json
```

第一次下載約 84.2 MiB 官方權重，並核對固定官方程式與權重 SHA-256。這只提取兩張受控圖的特徵，不訓練 DINOv2／DINOv3；輸入 PNG、NPZ 與 JSON 存在 artifacts/runs/，解讀見網頁。

In [1]:
"""A hand-specified Gram example, not DINOv3 training or a pretrained-model benchmark."""
import json

import torch
import torch.nn.functional as F


def main():
    torch.set_num_threads(2)
    anchor = F.normalize(torch.tensor([[1., 0.], [0., 1.], [1., 1.]]), dim=-1)
    rotation = torch.tensor([[0., -1.], [1., 0.]])
    rotated = anchor @ rotation
    constant = torch.tensor([[1., 0.], [1., 0.], [1., 0.]])
    anchor_gram = anchor @ anchor.T
    rotated_gram = rotated @ rotated.T
    constant_gram = constant @ constant.T
    rotated_gram_mse = (rotated_gram - anchor_gram).square().mean()
    constant_gram_mse = (constant_gram - anchor_gram).square().mean()
    feature_mse = (rotated - anchor).square().mean()
    assert torch.allclose(anchor_gram, rotated_gram, atol=1e-7)
    assert rotated_gram_mse == 0 and feature_mse > 0 and constant_gram_mse > 0
    assert torch.allclose(anchor.norm(dim=-1), torch.ones(3))
    assert torch.allclose(torch.diag(anchor_gram), torch.ones(3))
    print(json.dumps({
        "mode": "hand-specified normalized patch relations; no optimizer, no downloads",
        "anchor_features": anchor.tolist(), "anchor_gram": anchor_gram.tolist(),
        "rotated_feature_mse": float(feature_mse),
        "rotated_gram_mse": float(rotated_gram_mse),
        "constant_gram": constant_gram.tolist(),
        "constant_gram_mse": float(constant_gram_mse),
        "scope": "Mean squared Gram difference illustrates relation preservation; not the complete DINOv3 loss or schedule.",
    }, indent=2))


if __name__ == "__main__":
    main()


{
  "mode": "hand-specified normalized patch relations; no optimizer, no downloads",
  "anchor_features": [
    [
      1.0,
      0.0
    ],
    [
      0.0,
      1.0
    ],
    [
      0.7071067690849304,
      0.7071067690849304
    ]
  ],
  "anchor_gram": [
    [
      1.0,
      0.0,
      0.7071067690849304
    ],
    [
      0.0,
      1.0,
      0.7071067690849304
    ],
    [
      0.7071067690849304,
      0.7071067690849304,
      0.9999999403953552
    ]
  ],
  "rotated_feature_mse": 1.0,
  "rotated_gram_mse": 0.0,
  "constant_gram": [
    [
      1.0,
      1.0,
      1.0
    ],
    [
      1.0,
      1.0,
      1.0
    ],
    [
      1.0,
      1.0,
      1.0
    ]
  ],
  "constant_gram_mse": 0.26034948229789734,
  "scope": "Mean squared Gram difference illustrates relation preservation; not the complete DINOv3 loss or schedule."
}
